# cAPTure development OOF early-warning audit

This notebook reuses the completed operational OOF report, its selected score thresholds, and the checksum-validated development OOF packet predictions. It does not train models, select new thresholds, or access final-test scenarios.

The primary audit uses the frozen five-second window-close decision protocol. An attack-step iteration is timely only if the first qualifying window-close alert occurs strictly before its last malicious packet. For each development scenario, the first correct chain alert is early only if it occurs strictly before the first packet of a declared terminal action. Terminal actions are frozen in `configs/capture_early_warning_audit_v1.yaml`. Each scenario is one observed chain episode, so chain-level rates are descriptive.

A secondary score-availability sensitivity reads the OOF packet Parquet files and assigns each qualifying score to its packet timestamp instead of the five-second window close. This is a causal packet-arrival estimate for `xgb_p` and `history`, assuming negligible inference latency. The `current_window` and `full` models use complete-current-window summaries, so their packet-timestamp results are optimistic backdated diagnostics and cannot be interpreted as deployable early-warning performance without retraining with incrementally available context.

The selected thresholds and false-alert accounting remain unchanged. False alerts continue to be deduplicated and measured at the five-second window level.


## 1. Prepare Colab


In [1]:
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime, timezone
from pathlib import Path
import json
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
if not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", "--branch", REPOSITORY_BRANCH,
                    "--single-branch", REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
branch = subprocess.check_output(["git", "branch", "--show-current"],
                                 cwd=PROJECT_ROOT, text=True).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")
required_files = [
    "code/python/requirements-capture-xgb.txt",
    "code/python/utils/capture_early_warning.py",
    "code/python/utils/capture_oof_operational.py",
    "configs/capture_experiment_v1.yaml",
    "configs/capture_early_warning_audit_v1.yaml",
]
missing = [name for name in required_files if not (PROJECT_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Update the Colab repository copy first: {missing}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(PROJECT_ROOT / "code/python/requirements-capture-xgb.txt")], check=True)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))
import pandas as pd
from IPython.display import display
print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip())


Mounted at /content/drive
Repository commit: f2f8b1c2023a737bfc9618936e9f6b76e0a3a4dc


## 2. Select the completed operational run

The default ID is the operational run recorded in the previous notebook. Change it if your completed run has another ID. Set `EARLY_WARNING_RUN_ID` only when resuming an existing audit.


In [18]:
from utils.capture_early_warning import (
    run_early_warning_audit, validate_early_warning_audit,
)

MANIFEST_PATH = PROJECT_ROOT / "configs/capture_experiment_v1.yaml"
POLICY_PATH = PROJECT_ROOT / "configs/capture_early_warning_audit_v1.yaml"
OPERATIONAL_RUN_ID = "20260920T142514_611048Z_operational_oof"
EARLY_WARNING_RUN_ID = None  # Set only when resuming an existing audit.
if EARLY_WARNING_RUN_ID is None:
    EARLY_WARNING_RUN_ID = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
        + "_early_warning_oof"
    )
OPERATIONAL_DIR = DRIVE_ROOT / "operational_oof_runs" / OPERATIONAL_RUN_ID
OUTPUT_DIR = DRIVE_ROOT / "early_warning_oof_runs" / EARLY_WARNING_RUN_ID
print("Operational input:", OPERATIONAL_DIR)
print("Audit output:", OUTPUT_DIR)


Operational input: /content/drive/MyDrive/capture_gate0/operational_oof_runs/20260920T142514_611048Z_operational_oof
Audit output: /content/drive/MyDrive/capture_gate0/early_warning_oof_runs/20260922T141405_424398Z_early_warning_oof


## 3. Compute or verify the immutable audit


In [19]:
audit_args = {
    "operational_dir": OPERATIONAL_DIR,
    "manifest_path": MANIFEST_PATH,
    "policy_path": POLICY_PATH,
    "output_dir": OUTPUT_DIR,
}
if OUTPUT_DIR.exists():
    report = validate_early_warning_audit(**audit_args)
else:
    report = run_early_warning_audit(**audit_args)
operational = json.loads(
    (OPERATIONAL_DIR / "operational_report.json").read_text(encoding="utf-8")
)
print("Early-warning audit run ID:", EARLY_WARNING_RUN_ID)
print("Terminal actions:", report["policy"]["terminal_action_steps"])


Early-warning audit run ID: 20260922T141405_424398Z_early_warning_oof
Terminal actions: {'train_dollar_char': ['dollar_char'], 'train_slash_char': ['slash_char'], 'train_sub_exf': ['scp_exf'], 'train_empty_conn': ['empty_conn', 'empty_conn_ddos'], 'train_qos_mid': ['qos_mid', 'qos_mid_ddos']}


## 4. Compare warning times at every declared alert budget

The score-positive rate reproduces the original iteration detection rule. The timely rate requires the alert to arrive before the iteration's last malicious packet. The preterminal chain metric is a hierarchical macro of one episode-level flag per scenario; inspect the scenario table below rather than treating five episodes as a large sample.


In [20]:
comparison_rows = []
for model_name, model_report in report["models"].items():
    for budget_name in report["budget_order"]:
        audit_budget = model_report["budgets"][budget_name]
        operational_budget = operational["models"][model_name]["budgets"][budget_name]
        comparison_rows.append({
            "model": model_name,
            "budget": budget_name,
            "threshold": model_report["thresholds"][budget_name]["threshold"],
            "false_alert_windows_per_hour": (
                operational_budget["hierarchical_macro"]["false_alert_windows_per_hour"]),
            **audit_budget["hierarchical_macro"],
        })
comparison = pd.DataFrame(comparison_rows).set_index(["model", "budget"])
display(comparison)


threshold  false_alert_windows_per_hour  \
model          budget                                                       
xgb_p          one_per_hour        0.999463                      0.208772   
               one_per_12_hours    0.999942                      0.000000   
               one_per_5_minutes   0.987051                      3.163539   
current_window one_per_hour        0.983387                      0.658179   
               one_per_12_hours    0.994354                      0.000000   
               one_per_5_minutes   0.959686                      5.655613   
history        one_per_hour        0.995854                      0.918791   
               one_per_12_hours    0.999206                      0.036074   
               one_per_5_minutes   0.988180                      6.718528   
full           one_per_hour        0.997506                      0.433099   
               one_per_12_hours    0.998009                      0.036074   
               one_per_5_minutes   0.996420                      4.158372   

                                  score_positive_iteration_rate  \
model          budget                                             
xgb_p          one_per_hour                            0.528501   
               one_per_12_hours                        0.395452   
               one_per_5_minutes                       0.994527   
current_window one_per_hour                            0.909566   
               one_per_12_hours                        0.838935   
               one_per_5_minutes                       0.974778   
history        one_per_hour                            0.985104   
               one_per_12_hours                        0.860758   
               one_per_5_minutes                       0.995522   
full           one_per_hour                            0.844987   
               one_per_12_hours                        0.827461   
               one_per_5_minutes                       0.863420   

                                  timely_iteration_rate  \
model          budget                                     
xgb_p          one_per_hour                    0.395503   
               one_per_12_hours                0.281386   
               one_per_5_minutes               0.891673   
current_window one_per_hour                    0.847208   
               one_per_12_hours                0.824118   
               one_per_5_minutes               0.878904   
history        one_per_hour                    0.884298   
               one_per_12_hours                0.810621   
               one_per_5_minutes               0.894189   
full           one_per_hour                    0.827217   
               one_per_12_hours                0.817031   
               one_per_5_minutes               0.847334   

                                  late_positive_iteration_rate  \
model          budget                                            
xgb_p          one_per_hour                           0.132998   
               one_per_12_hours                       0.114067   
               one_per_5_minutes                      0.102855   
current_window one_per_hour                           0.062358   
               one_per_12_hours                       0.014817   
               one_per_5_minutes                      0.095874   
history        one_per_hour                           0.100806   
               one_per_12_hours                       0.050137   
               one_per_5_minutes                      0.101333   
full           one_per_hour                           0.017770   
               one_per_12_hours                       0.010430   
               one_per_5_minutes                      0.016087   

                                  early_before_terminal_action  
model          budget                                           
xgb_p          one_per_hour                                1.0  
               one_per_12_hours                            1.0  
    

## 5. Inspect individual scenarios and terminal-action deadlines


In [21]:
PRIMARY_BUDGET = "one_per_hour"
scenario_rows = []
chain_rows = []
for model_name, model_report in report["models"].items():
    audit_budget = model_report["budgets"][PRIMARY_BUDGET]
    for scenario, item in audit_budget["scenario_metrics"].items():
        scenario_rows.append({"model": model_name, **item})
    for scenario, item in audit_budget["chain_metrics"].items():
        chain_rows.append({"model": model_name, **item})
scenario_columns = [
    "fold", "iterations", "score_positive_iterations",
    "timely_iterations", "late_positive_iterations",
    "timely_iteration_rate", "early_before_terminal_action",
    "preterminal_opportunity_seconds",
]
display(pd.DataFrame(scenario_rows).set_index(["model", "scenario"])[scenario_columns])
chain_columns = [
    "first_terminal_action_step", "preterminal_opportunity_seconds",
    "first_correct_alert_steps", "early_before_terminal_action",
    "seconds_before_terminal_action",
    "seconds_at_or_after_terminal_action",
]
display(pd.DataFrame(chain_rows).set_index(["model", "scenario"])[chain_columns])


fold  iterations  score_positive_iterations  \
model          scenario                                                        
xgb_p          train_dollar_char    A         232                        172   
               train_slash_char     A         356                        295   
               train_sub_exf        A         335                        191   
               train_empty_conn     B         107                         46   
               train_qos_mid        B         171                         44   
current_window train_dollar_char    A         232                        226   
               train_slash_char     A         356                        355   
               train_sub_exf        A         335                        238   
               train_empty_conn     B         107                         96   
               train_qos_mid        B         171                        163   
history        train_dollar_char    A         232                        232   
               train_slash_char     A         356                        355   
               train_sub_exf        A         335                        306   
               train_empty_conn     B         107                        107   
               train_qos_mid        B         171                        171   
full           train_dollar_char    A         232                        220   
               train_slash_char     A         356                        352   
               train_sub_exf        A         335                        206   
               train_empty_conn     B         107                         82   
               train_qos_mid        B         171                        156   

                                  timely_iterations  late_positive_iterations  \
model          scenario                                                         
xgb_p          train_dollar_char                126                        46   
               train_slash_char                 251                        44   
               train_sub_exf                    102                        89   
               train_empty_conn                  41                         5   
               train_qos_mid                     28                        16   
current_window train_dollar_char                221                         5   
               train_slash_char                 348                         7   
               train_sub_exf                    194                        44   
               train_empty_conn                  86                        10   
               train_qos_mid                    156                         7   
history        train_dollar_char                225                         7   
               train_slash_char                 348                         7   
               train_sub_exf                    197                       109   
               train_empty_conn                  95                        12   
               train_qos_mid                    164                         7   
full           train_dollar_char                219                         1   
               train_slash_char                 348                         4   
               train_sub_exf                    199                         7   
               train_empty_conn                  82                         0   
               train_qos_mid                    148                         8   

                                  timely_iteration_rate  \
model          scenario                                   
xgb_p          train_dollar_char               0.543103   
               train_slash_char                0.705056   
               train_sub_exf                   0.304478   
               train_empty_conn                0.383178   
               train_qos_mid                   0.163743   
current_window train_dollar_char               0.952586   
               tr

first_terminal_action_step  \
model          scenario                                       
xgb_p          train_dollar_char                dollar_char   
               train_slash_char                  slash_char   
               train_sub_exf                        scp_exf   
               train_empty_conn                  empty_conn   
               train_qos_mid                        qos_mid   
current_window train_dollar_char                dollar_char   
               train_slash_char                  slash_char   
               train_sub_exf                        scp_exf   
               train_empty_conn                  empty_conn   
               train_qos_mid                        qos_mid   
history        train_dollar_char                dollar_char   
               train_slash_char                  slash_char   
               train_sub_exf                        scp_exf   
               train_empty_conn                  empty_conn   
               train_qos_mid                        qos_mid   
full           train_dollar_char                dollar_char   
               train_slash_char                  slash_char   
               train_sub_exf                        scp_exf   
               train_empty_conn                  empty_conn   
               train_qos_mid                        qos_mid   

                                  preterminal_opportunity_seconds  \
model          scenario                                             
xgb_p          train_dollar_char                      6718.998145   
               train_slash_char                       3534.824399   
               train_sub_exf                          3785.638018   
               train_empty_conn                       5555.421036   
               train_qos_mid                          3463.147562   
current_window train_dollar_char                      6718.998145   
               train_slash_char                       3534.824399   
               train_sub_exf                          3785.638018   
               train_empty_conn                       5555.421036   
               train_qos_mid                          3463.147562   
history        train_dollar_char                      6718.998145   
               train_slash_char                       3534.824399   
               train_sub_exf                          3785.638018   
               train_empty_conn                       5555.421036   
               train_qos_mid                          3463.147562   
full           train_dollar_char                      6718.998145   
               train_slash_char                       3534.824399   
               train_sub_exf                          3785.638018   
               train_empty_conn                       5555.421036   
               train_qos_mid                          3463.147562   

                                 first_correct_alert_steps  \
model          scenario                                      
xgb_p          train_dollar_char              [nmap_10_T5]   
               train_slash_char               [nmap_10_T5]   
               train_sub_exf                  [nmap_10_T5]   
               train_empty_conn               [nmap_10_T4]   
               train_qos_mid                 [nmap_banner]   
current_window train_dollar_char              [nmap_10_T5]   
               train_slash_char               [nmap_10_T5]   
               train_sub_exf                  [nmap_10_T5]   
               train_empty_conn               [nmap_10_T4]   
               train_qos_mid                 [nmap_banner]   
history        train_dollar_char              [nmap_10_T5]   
               train_slash_char               [nmap_10_T5]   
               train_sub_exf                  [nmap_10_T5]   
               train_empty_conn               [nmap_10_T4]   
               train_qos_mid                 [nmap_banner]   
full           train_dollar_char              [nmap_10_T5]   
           

## 6. Inspect attack steps with late or missing alerts


In [22]:
step_rows = []
for model_name, model_report in report["models"].items():
    for item in model_report["budgets"][PRIMARY_BUDGET]["step_metrics"].values():
        step_rows.append({"model": model_name, **item})
step_columns = [
    "model", "scenario", "attack_step", "iterations",
    "timely_iterations", "late_positive_iterations",
    "no_score_positive_iterations", "timely_iteration_rate",
]
step_table = pd.DataFrame(step_rows)[step_columns]
display(step_table.sort_values(
    ["timely_iteration_rate", "iterations"], ascending=[True, False]
).head(40))
print("Complete step and iteration details are stored in:",
      OUTPUT_DIR / "early_warning_report.json")


,model,scenario,attack_step,iterations,timely_iterations,late_positive_iterations,no_score_positive_iterations,timely_iteration_rate
23,xgb_p,train_sub_exf,scp_exf,117,0,0,117,0.000000
65,current_window,train_sub_exf,scp_exf,117,0,41,76,0.000000
39,xgb_p,train_qos_mid,qos_mid,39,0,0,39,0.000000
25,xgb_p,train_empty_conn,empty_conn,37,0,0,37,0.000000
18,xgb_p,train_sub_exf,mqtt_cat,27,0,0,27,0.000000
27,xgb_p,train_empty_conn,mqtt_cat,12,0,0,12,0.000000
32,xgb_p,train_empty_conn,sftp_inst,12,0,0,12,0.000000
74,current_window,train_empty_conn,sftp_inst,12,0,8,4,0.000000
116,history,train_empty_conn,sftp_inst,12,0,12,0,0.000000
158,full,train_empty_conn,sftp_inst,12,0,0,12,0.000000


Complete step and iteration details are stored in: /content/drive/MyDrive/capture_gate0/early_warning_oof_runs/20260922T141405_424398Z_early_warning_oof/early_warning_report.json


## 7. Packet-time score-availability sensitivity

This sensitivity separates model coverage from the delay introduced by waiting for the five-second window to close. It preserves the existing model-specific thresholds and window-level false-alert accounting.

For `xgb_p` and `history`, the packet-timestamp result is causal under the assumption that inference latency is negligible. For `current_window` and `full`, it is an optimistic diagnostic because their scores depend on summaries of the complete current window.


In [23]:
import duckdb

from utils.capture_data import load_manifest
from utils.capture_oof_operational import (
    MODEL_NAMES,
    _scenario_oof_path,
    _validated_runs,
)


SCORE_AVAILABILITY_ASSUMPTIONS = {
    "xgb_p": "causal_packet_arrival",
    "history": "causal_packet_arrival",
    "current_window": "optimistic_backdated_complete_window_context",
    "full": "optimistic_backdated_complete_window_context",
}


def summarize_packet_timestamp_iterations(
    oof_path: Path,
    threshold: float,
) -> pd.DataFrame:
    """Measure iteration timeliness when a score is available at packet time."""
    connection = duckdb.connect()
    try:
        connection.execute("SET threads = 2")
        connection.execute("SET memory_limit = '4GB'")
        result = connection.execute(
            """
            WITH attack_packets AS (
                SELECT
                    attack_step,
                    sequence_id,
                    packet_timestamp_ns,
                    CAST(score AS DOUBLE) AS score
                FROM read_parquet(?)
                WHERE binary_label = 1
            ),
            iterations AS (
                SELECT
                    attack_step,
                    sequence_id,
                    MIN(packet_timestamp_ns) AS first_malicious_packet_ns,
                    MAX(packet_timestamp_ns) AS last_malicious_packet_ns,
                    MIN(
                        CASE
                            WHEN score >= ? THEN packet_timestamp_ns
                            ELSE NULL
                        END
                    ) AS first_score_positive_packet_ns
                FROM attack_packets
                GROUP BY attack_step, sequence_id
            )
            SELECT
                attack_step,
                CAST(COUNT(*) AS BIGINT) AS iterations,
                CAST(
                    COUNT(first_score_positive_packet_ns)
                    AS BIGINT
                ) AS score_positive_iterations,
                CAST(
                    SUM(
                        CASE
                            WHEN first_score_positive_packet_ns
                                  < last_malicious_packet_ns
                            THEN 1 ELSE 0
                        END
                    )
                    AS BIGINT
                ) AS timely_iterations,
                CAST(
                    SUM(
                        CASE
                            WHEN first_score_positive_packet_ns
                                  = last_malicious_packet_ns
                            THEN 1 ELSE 0
                        END
                    )
                    AS BIGINT
                ) AS late_positive_iterations,
                CAST(
                    SUM(
                        CASE
                            WHEN first_score_positive_packet_ns IS NULL
                            THEN 1 ELSE 0
                        END
                    )
                    AS BIGINT
                ) AS no_score_positive_iterations
            FROM iterations
            GROUP BY attack_step
            ORDER BY attack_step
            """,
            [str(oof_path), float(threshold)],
        ).fetchdf()
    finally:
        connection.close()

    count_columns = [
        "iterations",
        "score_positive_iterations",
        "timely_iterations",
        "late_positive_iterations",
        "no_score_positive_iterations",
    ]
    for column in count_columns:
        result[column] = result[column].astype("int64")

    return result


manifest = load_manifest(MANIFEST_PATH)
run_dirs = {
    model_name: Path(run_path)
    for model_name, run_path in operational["input_runs"].items()
}
fold_reports = _validated_runs(manifest, run_dirs)

packet_timestamp_rows = []

for model_name in MODEL_NAMES:
    threshold = float(
        report["models"][model_name]["thresholds"][PRIMARY_BUDGET]["threshold"]
    )
    window_step_metrics = report["models"][model_name]["budgets"][
        PRIMARY_BUDGET
    ]["step_metrics"]

    for fold, split in manifest["validation"]["folds"].items():
        fold_report = fold_reports[model_name][fold]

        for scenario in split["validate"]:
            oof_path = _scenario_oof_path(
                run_dirs[model_name],
                fold,
                fold_report,
                scenario,
            )
            packet_metrics = summarize_packet_timestamp_iterations(
                oof_path,
                threshold,
            )

            for item in packet_metrics.to_dict("records"):
                attack_step = item["attack_step"]
                metric_key = f"{scenario}::{attack_step}"
                window_metrics = window_step_metrics[metric_key]

                if (
                    int(item["iterations"])
                    != int(window_metrics["iterations"])
                    or int(item["score_positive_iterations"])
                    != int(window_metrics["score_positive_iterations"])
                ):
                    raise ValueError(
                        "Packet-time counts differ from the existing "
                        f"audit for {model_name}/{scenario}/{attack_step}."
                    )

                packet_timestamp_rows.append({
                    "model": model_name,
                    "scenario": scenario,
                    "fold": fold,
                    "attack_step": attack_step,
                    "threshold": threshold,
                    "score_availability_assumption": (
                        SCORE_AVAILABILITY_ASSUMPTIONS[model_name]
                    ),
                    "iterations": int(item["iterations"]),
                    "score_positive_iterations": int(
                        item["score_positive_iterations"]
                    ),
                    "window_close_timely_iterations": int(
                        window_metrics["timely_iterations"]
                    ),
                    "packet_timestamp_timely_iterations": int(
                        item["timely_iterations"]
                    ),
                    "packet_timestamp_late_positive_iterations": int(
                        item["late_positive_iterations"]
                    ),
                    "no_score_positive_iterations": int(
                        item["no_score_positive_iterations"]
                    ),
                })


packet_timestamp_step_table = pd.DataFrame(packet_timestamp_rows)

packet_timestamp_step_table[
    "window_close_timely_iteration_rate"
] = (
    packet_timestamp_step_table["window_close_timely_iterations"]
    / packet_timestamp_step_table["iterations"]
)
packet_timestamp_step_table[
    "packet_timestamp_timely_iteration_rate"
] = (
    packet_timestamp_step_table["packet_timestamp_timely_iterations"]
    / packet_timestamp_step_table["iterations"]
)
packet_timestamp_step_table[
    "timely_iteration_rate_gain"
] = (
    packet_timestamp_step_table[
        "packet_timestamp_timely_iteration_rate"
    ]
    - packet_timestamp_step_table[
        "window_close_timely_iteration_rate"
    ]
)

step_display_columns = [
    "model",
    "scenario",
    "attack_step",
    "score_availability_assumption",
    "iterations",
    "score_positive_iterations",
    "window_close_timely_iterations",
    "packet_timestamp_timely_iterations",
    "packet_timestamp_late_positive_iterations",
    "no_score_positive_iterations",
    "window_close_timely_iteration_rate",
    "packet_timestamp_timely_iteration_rate",
    "timely_iteration_rate_gain",
]

display(
    packet_timestamp_step_table[step_display_columns]
    .sort_values(
        [
            "packet_timestamp_timely_iteration_rate",
            "iterations",
        ],
        ascending=[True, False],
    )
    .head(40)
)


,model,scenario,attack_step,score_availability_assumption,iterations,score_positive_iterations,window_close_timely_iterations,packet_timestamp_timely_iterations,packet_timestamp_late_positive_iterations,no_score_positive_iterations,window_close_timely_iteration_rate,packet_timestamp_timely_iteration_rate,timely_iteration_rate_gain
23,xgb_p,train_sub_exf,scp_exf,causal_packet_arrival,117,0,0,0,0,117,0.000000,0.000000,0.000000
39,xgb_p,train_qos_mid,qos_mid,causal_packet_arrival,39,0,0,0,0,39,0.000000,0.000000,0.000000
25,xgb_p,train_empty_conn,empty_conn,causal_packet_arrival,37,0,0,0,0,37,0.000000,0.000000,0.000000
18,xgb_p,train_sub_exf,mqtt_cat,causal_packet_arrival,27,0,0,0,0,27,0.000000,0.000000,0.000000
27,xgb_p,train_empty_conn,mqtt_cat,causal_packet_arrival,12,0,0,0,0,12,0.000000,0.000000,0.000000
32,xgb_p,train_empty_conn,sftp_inst,causal_packet_arrival,12,0,0,0,0,12,0.000000,0.000000,0.000000
158,full,train_empty_conn,sftp_inst,optimistic_backdated_complete_window_context,12,0,0,0,0,12,0.000000,0.000000,0.000000
34,xgb_p,train_qos_mid,mqtt_cat,causal_packet_arrival,10,0,0,0,0,10,0.000000,0.000000,0.000000
133,full,train_dollar_char,scp_inst,optimistic_backdated_complete_window_context,7,0,0,0,0,7,0.000000,0.000000,0.000000
41,xgb_p,train_qos_mid,scp_inst,causal_packet_arrival,5,0,0,0,0,5,0.000000,0.000000,0.000000


In [24]:
scenario_count_columns = [
    "iterations",
    "score_positive_iterations",
    "window_close_timely_iterations",
    "packet_timestamp_timely_iterations",
    "packet_timestamp_late_positive_iterations",
    "no_score_positive_iterations",
]

packet_timestamp_scenario_table = (
    packet_timestamp_step_table
    .groupby(
        [
            "model",
            "scenario",
            "fold",
            "threshold",
            "score_availability_assumption",
        ],
        as_index=False,
        sort=False,
    )[scenario_count_columns]
    .sum()
)

packet_timestamp_scenario_table[
    "score_positive_iteration_rate"
] = (
    packet_timestamp_scenario_table["score_positive_iterations"]
    / packet_timestamp_scenario_table["iterations"]
)
packet_timestamp_scenario_table[
    "window_close_timely_iteration_rate"
] = (
    packet_timestamp_scenario_table["window_close_timely_iterations"]
    / packet_timestamp_scenario_table["iterations"]
)
packet_timestamp_scenario_table[
    "packet_timestamp_timely_iteration_rate"
] = (
    packet_timestamp_scenario_table[
        "packet_timestamp_timely_iterations"
    ]
    / packet_timestamp_scenario_table["iterations"]
)
packet_timestamp_scenario_table[
    "timely_iteration_rate_gain"
] = (
    packet_timestamp_scenario_table[
        "packet_timestamp_timely_iteration_rate"
    ]
    - packet_timestamp_scenario_table[
        "window_close_timely_iteration_rate"
    ]
)

scenario_display_columns = [
    "model",
    "scenario",
    "fold",
    "score_availability_assumption",
    "iterations",
    "score_positive_iterations",
    "window_close_timely_iterations",
    "packet_timestamp_timely_iterations",
    "packet_timestamp_late_positive_iterations",
    "no_score_positive_iterations",
    "window_close_timely_iteration_rate",
    "packet_timestamp_timely_iteration_rate",
    "timely_iteration_rate_gain",
]

display(
    packet_timestamp_scenario_table[scenario_display_columns]
    .set_index(["model", "scenario"])
)

rate_columns = [
    "window_close_timely_iteration_rate",
    "packet_timestamp_timely_iteration_rate",
    "timely_iteration_rate_gain",
]

packet_timestamp_fold_table = (
    packet_timestamp_scenario_table
    .groupby(
        ["model", "fold", "score_availability_assumption"],
        as_index=False,
        sort=False,
    )[rate_columns]
    .mean()
)

packet_timestamp_macro_table = (
    packet_timestamp_fold_table
    .groupby(
        ["model", "score_availability_assumption"],
        as_index=False,
        sort=False,
    )[rate_columns]
    .mean()
    .set_index("model")
)

display(packet_timestamp_macro_table)



fold  \
model          scenario                 
xgb_p          train_dollar_char    A   
               train_slash_char     A   
               train_sub_exf        A   
               train_empty_conn     B   
               train_qos_mid        B   
current_window train_dollar_char    A   
               train_slash_char     A   
               train_sub_exf        A   
               train_empty_conn     B   
               train_qos_mid        B   
history        train_dollar_char    A   
               train_slash_char     A   
               train_sub_exf        A   
               train_empty_conn     B   
               train_qos_mid        B   
full           train_dollar_char    A   
               train_slash_char     A   
               train_sub_exf        A   
               train_empty_conn     B   
               train_qos_mid        B   

                                                 score_availability_assumption  \
model          scenario                                                          
xgb_p          train_dollar_char                         causal_packet_arrival   
               train_slash_char                          causal_packet_arrival   
               train_sub_exf                             causal_packet_arrival   
               train_empty_conn                          causal_packet_arrival   
               train_qos_mid                             causal_packet_arrival   
current_window train_dollar_char  optimistic_backdated_complete_window_context   
               train_slash_char   optimistic_backdated_complete_window_context   
               train_sub_exf      optimistic_backdated_complete_window_context   
               train_empty_conn   optimistic_backdated_complete_window_context   
               train_qos_mid      optimistic_backdated_complete_window_context   
history        train_dollar_char                         causal_packet_arrival   
               train_slash_char                          causal_packet_arrival   
               train_sub_exf                             causal_packet_arrival   
               train_empty_conn                          causal_packet_arrival   
               train_qos_mid                             causal_packet_arrival   
full           train_dollar_char  optimistic_backdated_complete_window_context   
               train_slash_char   optimistic_backdated_complete_window_context   
               train_sub_exf      optimistic_backdated_complete_window_context   
               train_empty_conn   optimistic_backdated_complete_window_context   
               train_qos_mid      optimistic_backdated_complete_window_context   

                                  iterations  score_positive_iterations  \
model          scenario                                                   
xgb_p          train_dollar_char         232                        172   
               train_slash_char          356                        295   
               train_sub_exf             335                        191   
               train_empty_conn          107                         46   
               train_qos_mid             171                         44   
current_window train_dollar_char         232                        226   
               train_slash_char          356                        355   
               train_sub_exf             335                        238   
               train_empty_conn          107                         96   
               train_qos_mid             171                        163   
history        train_dollar_char         232                        232   
               train_slash_char          356                        355   
               train_sub_exf             335                        306   
               train_empty_conn          107                        107   
               train_qos_mid             171                        171   
full           train_dollar_char         232      

,score_availability_assumption,window_close_timely_iteration_rate,packet_timestamp_timely_iteration_rate,timely_iteration_rate_gain
model,,,,
xgb_p,causal_packet_arrival,0.395503,0.528501,0.132998
current_window,optimistic_backdated_complete_window_context,0.847208,0.909566,0.062358
history,causal_packet_arrival,0.884298,0.985104,0.100806
full,optimistic_backdated_complete_window_context,0.827217,0.844987,0.017770


## 8. Sensitivity analysis excluding Nmap steps

Nmap steps may provide unusually distinctive reconnaissance signals. This sensitivity analysis
removes every attack-step iteration whose name starts with `nmap` and recomputes the hierarchical
macro detection and timely-detection rates.

It also recomputes chain-level early warning while preventing Nmap detections from serving as the
first correct chain alert. False-alert rates and thresholds remain unchanged because benign traffic
is not removed or relabeled.

The analysis uses the score availability recorded by the immutable early-warning report. Therefore,
`timely_iteration_rate` refers to window-close availability, while `score_positive_iteration_rate`
only asks whether an iteration contains at least one score-positive packet.

In [27]:
NMAP_STEP_PREFIX = "nmap"


def is_nmap_attack_step(attack_step):
    return str(attack_step).strip().casefold().startswith(NMAP_STEP_PREFIX)


def hierarchical_macro_from_scenarios(scenario_table, metric_columns):
    fold_means = (
        scenario_table
        .groupby("fold", sort=True)[metric_columns]
        .mean()
    )
    return fold_means.mean(), fold_means


non_nmap_comparison_rows = []
non_nmap_scenario_rows = []
non_nmap_chain_rows = []

for model_name, model_report in report["models"].items():
    for budget_name in report["budget_order"]:
        audit_budget = model_report["budgets"][budget_name]
        operational_budget = operational["models"][model_name]["budgets"][budget_name]

        iteration_rows = audit_budget["iteration_rows"]
        excluded_rows = [
            item for item in iteration_rows
            if is_nmap_attack_step(item["attack_step"])
        ]
        retained_rows = [
            item for item in iteration_rows
            if not is_nmap_attack_step(item["attack_step"])
        ]

        if not retained_rows:
            raise ValueError(
                f"No non-Nmap iterations remain for {model_name}/{budget_name}."
            )

        non_nmap_iteration_table = pd.DataFrame(retained_rows)
        boolean_columns = [
            "score_positive",
            "timely_before_last_packet",
            "late_at_or_after_last_packet",
        ]
        for column in boolean_columns:
            non_nmap_iteration_table[column] = non_nmap_iteration_table[column].astype(bool)

        scenario_table = (
            non_nmap_iteration_table
            .groupby(["fold", "scenario"], sort=True)
            .agg(
                iterations=("sequence_id", "size"),
                score_positive_iterations=("score_positive", "sum"),
                timely_iterations=("timely_before_last_packet", "sum"),
                late_positive_iterations=("late_at_or_after_last_packet", "sum"),
            )
            .reset_index()
        )

        scenario_table["no_score_positive_iterations"] = (
            scenario_table["iterations"]
            - scenario_table["score_positive_iterations"]
        )
        scenario_table["score_positive_iteration_rate"] = (
            scenario_table["score_positive_iterations"]
            / scenario_table["iterations"]
        )
        scenario_table["timely_iteration_rate"] = (
            scenario_table["timely_iterations"]
            / scenario_table["iterations"]
        )
        scenario_table["late_positive_iteration_rate"] = (
            scenario_table["late_positive_iterations"]
            / scenario_table["iterations"]
        )

        metric_columns = [
            "score_positive_iteration_rate",
            "timely_iteration_rate",
            "late_positive_iteration_rate",
        ]
        non_nmap_macro, non_nmap_fold_means = (
            hierarchical_macro_from_scenarios(
                scenario_table,
                metric_columns,
            )
        )

        original_macro = audit_budget["hierarchical_macro"]

        non_nmap_comparison_rows.append({
            "model": model_name,
            "budget": budget_name,
            "threshold": model_report["thresholds"][budget_name]["threshold"],
            "false_alert_windows_per_hour": (
                operational_budget["hierarchical_macro"][
                    "false_alert_windows_per_hour"
                ]
            ),
            "excluded_nmap_iterations": len(excluded_rows),
            "retained_non_nmap_iterations": len(retained_rows),
            "original_score_positive_iteration_rate": (
                original_macro["score_positive_iteration_rate"]
            ),
            "non_nmap_score_positive_iteration_rate": (
                non_nmap_macro["score_positive_iteration_rate"]
            ),
            "score_positive_rate_change": (
                non_nmap_macro["score_positive_iteration_rate"]
                - original_macro["score_positive_iteration_rate"]
            ),
            "original_window_close_timely_iteration_rate": (
                original_macro["timely_iteration_rate"]
            ),
            "non_nmap_window_close_timely_iteration_rate": (
                non_nmap_macro["timely_iteration_rate"]
            ),
            "window_close_timely_rate_change": (
                non_nmap_macro["timely_iteration_rate"]
                - original_macro["timely_iteration_rate"]
            ),
        })

        if budget_name == PRIMARY_BUDGET:
            for row in scenario_table.to_dict(orient="records"):
                non_nmap_scenario_rows.append({
                    "model": model_name,
                    **row,
                })

        chain_scenario_rows = []

        for scenario, scenario_items in non_nmap_iteration_table.groupby(
            "scenario",
            sort=True,
        ):
            scenario_records = scenario_items.to_dict(orient="records")
            chain_reference = audit_budget["chain_metrics"][scenario]
            terminal_onset_ns = int(
                chain_reference["terminal_action_first_malicious_packet_ns"]
            )

            positive_records = [
                item for item in scenario_records
                if item["score_positive"]
                and item["first_correct_alert_ns"] is not None
            ]

            if positive_records:
                first_alert_ns = min(
                    int(item["first_correct_alert_ns"])
                    for item in positive_records
                )
                first_alert_steps = sorted({
                    str(item["attack_step"])
                    for item in positive_records
                    if int(item["first_correct_alert_ns"]) == first_alert_ns
                })
            else:
                first_alert_ns = None
                first_alert_steps = []

            early_without_nmap = (
                first_alert_ns is not None
                and first_alert_ns < terminal_onset_ns
            )

            chain_row = {
                "model": model_name,
                "budget": budget_name,
                "fold": str(scenario_records[0]["fold"]),
                "scenario": scenario,
                "first_non_nmap_alert_steps": first_alert_steps,
                "first_non_nmap_alert_ns": first_alert_ns,
                "early_before_terminal_action_without_nmap": (
                    early_without_nmap
                ),
                "seconds_before_terminal_action_without_nmap": (
                    (terminal_onset_ns - first_alert_ns) / 1_000_000_000
                    if early_without_nmap
                    else None
                ),
                "seconds_at_or_after_terminal_action_without_nmap": (
                    (first_alert_ns - terminal_onset_ns) / 1_000_000_000
                    if first_alert_ns is not None
                    and not early_without_nmap
                    else None
                ),
            }
            chain_scenario_rows.append(chain_row)

            if budget_name == PRIMARY_BUDGET:
                non_nmap_chain_rows.append(chain_row)

        chain_scenario_table = pd.DataFrame(chain_scenario_rows)
        non_nmap_chain_macro, _ = hierarchical_macro_from_scenarios(
            chain_scenario_table,
            ["early_before_terminal_action_without_nmap"],
        )

        non_nmap_comparison_rows[-1].update({
            "original_early_before_terminal_action": (
                original_macro["early_before_terminal_action"]
            ),
            "non_nmap_early_before_terminal_action": (
                non_nmap_chain_macro[
                    "early_before_terminal_action_without_nmap"
                ]
            ),
            "early_warning_rate_change": (
                non_nmap_chain_macro[
                    "early_before_terminal_action_without_nmap"
                ]
                - original_macro["early_before_terminal_action"]
            ),
        })


non_nmap_comparison = (
    pd.DataFrame(non_nmap_comparison_rows)
    .set_index(["model", "budget"])
)

comparison_columns = [
    "threshold",
    "false_alert_windows_per_hour",
    "excluded_nmap_iterations",
    "retained_non_nmap_iterations",
    "original_score_positive_iteration_rate",
    "non_nmap_score_positive_iteration_rate",
    "score_positive_rate_change",
    "original_window_close_timely_iteration_rate",
    "non_nmap_window_close_timely_iteration_rate",
    "window_close_timely_rate_change",
    "original_early_before_terminal_action",
    "non_nmap_early_before_terminal_action",
    "early_warning_rate_change",
]

display(non_nmap_comparison[comparison_columns])


threshold  false_alert_windows_per_hour  \
model          budget                                                       
xgb_p          one_per_hour        0.999463                      0.208772   
               one_per_12_hours    0.999942                      0.000000   
               one_per_5_minutes   0.987051                      3.163539   
current_window one_per_hour        0.983387                      0.658179   
               one_per_12_hours    0.994354                      0.000000   
               one_per_5_minutes   0.959686                      5.655613   
history        one_per_hour        0.995854                      0.918791   
               one_per_12_hours    0.999206                      0.036074   
               one_per_5_minutes   0.988180                      6.718528   
full           one_per_hour        0.997506                      0.433099   
               one_per_12_hours    0.998009                      0.036074   
               one_per_5_minutes   0.996420                      4.158372   

                                  excluded_nmap_iterations  \
model          budget                                        
xgb_p          one_per_hour                            427   
               one_per_12_hours                        427   
               one_per_5_minutes                       427   
current_window one_per_hour                            427   
               one_per_12_hours                        427   
               one_per_5_minutes                       427   
history        one_per_hour                            427   
               one_per_12_hours                        427   
               one_per_5_minutes                       427   
full           one_per_hour                            427   
               one_per_12_hours                        427   
               one_per_5_minutes                       427   

                                  retained_non_nmap_iterations  \
model          budget                                            
xgb_p          one_per_hour                                774   
               one_per_12_hours                            774   
               one_per_5_minutes                           774   
current_window one_per_hour                                774   
               one_per_12_hours                            774   
               one_per_5_minutes                           774   
history        one_per_hour                                774   
               one_per_12_hours                            774   
               one_per_5_minutes                           774   
full           one_per_hour                                774   
               one_per_12_hours                            774   
               one_per_5_minutes                           774   

                                  original_score_positive_iteration_rate  \
model          budget                                                      
xgb_p          one_per_hour                                     0.528501   
               one_per_12_hours                                 0.395452   
               one_per_5_minutes                                0.994527   
current_window one_per_hour                                     0.909566   
               one_per_12_hours                                 0.838935   
               one_per_5_minutes                                0.974778   
history        one_per_hour                                     0.985104   
               one_per_12_hours                                 0.860758   
               one_per_5_minutes                                0.995522   
full           one_per_hour                                     0.844987   
               one_per_12_hours                                 0.827461   
               one_per_5_minutes                                0.863420   

                                  non_nmap_score_positive_iteration_rate  \
model          budge

In [28]:
non_nmap_scenario_table = (
    pd.DataFrame(non_nmap_scenario_rows)
    .set_index(["model", "scenario"])
)

display(non_nmap_scenario_table[[
    "fold",
    "iterations",
    "score_positive_iterations",
    "timely_iterations",
    "late_positive_iterations",
    "no_score_positive_iterations",
    "score_positive_iteration_rate",
    "timely_iteration_rate",
]])


non_nmap_chain_table = (
    pd.DataFrame(non_nmap_chain_rows)
    .set_index(["model", "scenario"])
)

display(non_nmap_chain_table[[
    "fold",
    "first_non_nmap_alert_steps",
    "early_before_terminal_action_without_nmap",
    "seconds_before_terminal_action_without_nmap",
    "seconds_at_or_after_terminal_action_without_nmap",
]])


fold  iterations  score_positive_iterations  \
model          scenario                                                        
xgb_p          train_dollar_char    A         181                        121   
               train_slash_char     A         248                        187   
               train_sub_exf        A         145                          1   
               train_empty_conn     B          69                          8   
               train_qos_mid        B         131                          4   
current_window train_dollar_char    A         181                        175   
               train_slash_char     A         248                        247   
               train_sub_exf        A         145                         48   
               train_empty_conn     B          69                         58   
               train_qos_mid        B         131                        123   
history        train_dollar_char    A         181                        181   
               train_slash_char     A         248                        247   
               train_sub_exf        A         145                        116   
               train_empty_conn     B          69                         69   
               train_qos_mid        B         131                        131   
full           train_dollar_char    A         181                        169   
               train_slash_char     A         248                        244   
               train_sub_exf        A         145                         16   
               train_empty_conn     B          69                         44   
               train_qos_mid        B         131                        116   

                                  timely_iterations  late_positive_iterations  \
model          scenario                                                         
xgb_p          train_dollar_char                 82                        39   
               train_slash_char                 180                         7   
               train_sub_exf                      1                         0   
               train_empty_conn                   8                         0   
               train_qos_mid                      4                         0   
current_window train_dollar_char                170                         5   
               train_slash_char                 240                         7   
               train_sub_exf                      4                        44   
               train_empty_conn                  48                        10   
               train_qos_mid                    116                         7   
history        train_dollar_char                174                         7   
               train_slash_char                 240                         7   
               train_sub_exf                     11                       105   
               train_empty_conn                  57                        12   
               train_qos_mid                    124                         7   
full           train_dollar_char                169                         0   
               train_slash_char                 240                         4   
               train_sub_exf                      9                         7   
               train_empty_conn                  44                         0   
               train_qos_mid                    108                         8   

                                  no_score_positive_iterations  \
model          scenario                                          
xgb_p          train_dollar_char                            60   
               train_slash_char                             61   
               train_sub_exf                               144   
               train_empty_conn                             61   
               train_qos_mid                               127   
current_window train_dollar

fold first_non_nmap_alert_steps  \
model          scenario                                            
xgb_p          train_dollar_char    A    [brute_force_malformed]   
               train_empty_conn     B       [brute_force_timing]   
               train_qos_mid        B    [brute_force_malformed]   
               train_slash_char     A       [brute_force_timing]   
               train_sub_exf        A       [brute_force_timing]   
current_window train_dollar_char    A    [brute_force_malformed]   
               train_empty_conn     B       [brute_force_timing]   
               train_qos_mid        B    [brute_force_malformed]   
               train_slash_char     A       [brute_force_timing]   
               train_sub_exf        A       [brute_force_timing]   
history        train_dollar_char    A    [brute_force_malformed]   
               train_empty_conn     B       [brute_force_timing]   
               train_qos_mid        B    [brute_force_malformed]   
               train_slash_char     A       [brute_force_timing]   
               train_sub_exf        A       [brute_force_timing]   
full           train_dollar_char    A    [brute_force_malformed]   
               train_empty_conn     B       [brute_force_timing]   
               train_qos_mid        B    [brute_force_malformed]   
               train_slash_char     A       [brute_force_timing]   
               train_sub_exf        A       [brute_force_timing]   

                                  early_before_terminal_action_without_nmap  \
model          scenario                                                       
xgb_p          train_dollar_char                                       True   
               train_empty_conn                                        True   
               train_qos_mid                                           True   
               train_slash_char                                        True   
               train_sub_exf                                           True   
current_window train_dollar_char                                       True   
               train_empty_conn                                        True   
               train_qos_mid                                           True   
               train_slash_char                                        True   
               train_sub_exf                                           True   
history        train_dollar_char                                       True   
               train_empty_conn                                        True   
               train_qos_mid                                           True   
               train_slash_char                                        True   
               train_sub_exf                                           True   
full           train_dollar_char                                       True   
               train_empty_conn                                        True   
               train_qos_mid                                           True   
               train_slash_char                                        True   
               train_sub_exf                                           True   

                                  seconds_before_terminal_action_without_nmap  \
model          scenario                                                         
xgb_p          train_dollar_char                                  5150.929966   
               train_empty_conn                                   4459.590510   
               train_qos_mid                                      2672.522326   
               train_slash_char                                   3109.974252   
               train_sub_exf                                      3046.753496   
current_window train_dollar_char                                  5235.929966   
               train_empty_conn                                   4549.590510   
               train_qos_mid                                      27

## Interpretation

The window-close audit measures performance under the frozen aligned five-second decision protocol. The packet-time sensitivity shows that the window boundary accounts for all observed late positive iterations when stored scores are assigned to packet timestamps.

For `xgb_p` and `history`, packet-time availability is causal under negligible inference latency. For `current_window` and `full`, packet-time availability is an optimistic backdated diagnostic because their scores use complete-current-window summaries. A causal packet-time version of those models would require incremental context construction and retraining.

Removing the window-close delay cannot recover iterations whose malicious packets never exceed the selected threshold. The selected thresholds still come from development OOF predictions, false alerts remain deduplicated at the five-second window level, and final-test performance remains unmeasured.

